# Data Cleaning for CIS 5450 Final Project
**By:** Alan Lin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('mlb_pitches_2026.csv')
display(df.head())
print('Number of pitches in our dataset:', len(df))

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,strikeout,swinging_strike,...,NaN,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,33.513220,63.065029
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,NaN,swinging_strike,...,NaN,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,25.242091,21.609779
2,FF,2026-09-27,95.4,-2.26,5.46,"Imai, Tatsuya",672016,837227,NaN,called_strike,...,NaN,1.41,1.11,1.11,25.6,NaN,NaN,NaN,NaN,NaN
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,field_out,hit_into_play,...,NaN,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,35.570114,45.381904
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,field_out,hit_into_play,...,NaN,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,44.436093,32.564172


Number of pitches in our dataset: 716792


In [3]:
clean_df = df.copy()
clean_df['description'].value_counts()

description
ball                       240992
foul                       129126
hit_into_play              124620
called_strike              115698
swinging_strike             73737
blocked_ball                14894
foul_tip                     7514
swinging_strike_blocked      3908
automatic_ball               2217
hit_by_pitch                 2119
foul_bunt                    1551
missed_bunt                   256
automatic_strike               92
pitchout                       39
bunt_foul_tip                  28
swinging_pitchout               1
Name: count, dtype: int64

In [4]:
# -- BUNTS EXCLUDED --
whiff_cases = ['swinging_strike', 'swinging_strike_blocked']
contact_cases = ['foul', 'hit_into_play', 'foul_tip']

In [5]:
# -- BUNTS INCLUDED --
# whiff_cases = ['swinging_strike', 'swinging_strike_blocked', 'missed_bunt', 'swinging_pitchout']
# contact_cases = ['foul', 'hit_into_play', 'foul_tip', 'foul_bunt', 'bunt_foul_tip']

With the bunt cases, we have $73737+3908+256+1 = 77902$ whiffs and $129126+124620+7514+1551+28 = 262839$ contact cases, totalling $340741$ pitches.

Without the bunts, we have $73737+3908 = 77645$ whiffs and $129126+124620+7514 = 261260$ contact cases, totalling $338905$ pitches.

Without the bunts, we lose about $1551+256+28+1 = 1836$ total pitches, which is about $0.5\%$ of our dataset.

In [6]:
clean_df = clean_df[clean_df['description'].isin(whiff_cases + contact_cases)]
clean_df['is_whiff'] = clean_df['description'].isin(whiff_cases).astype(int)
clean_df

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,is_whiff
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,strikeout,swinging_strike,...,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,33.513220,63.065029,1
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,NaN,swinging_strike,...,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,25.242091,21.609779,1
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,field_out,hit_into_play,...,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,35.570114,45.381904,0
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,field_out,hit_into_play,...,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,44.436093,32.564172,0
5,SL,2026-09-27,83.6,-2.27,5.87,"Imai, Tatsuya",695720,837227,NaN,foul,...,3.34,-0.05,-0.05,31.3,22.800268,-11.648310,37.518572,32.584465,36.323408,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
716781,SI,2026-03-25,92.0,-1.72,4.98,"Webb, Logan",592450,657277,NaN,foul,...,2.76,1.38,1.38,20.1,12.555138,-3.603324,36.148716,35.063382,26.389487,0
716786,FF,2026-03-25,93.6,-1.67,5.07,"Webb, Logan",663757,657277,strikeout,swinging_strike,...,1.81,0.93,-0.93,22.0,15.601773,6.257514,29.116399,34.023841,19.836725,1
716787,CH,2026-03-25,86.9,-1.97,4.78,"Webb, Logan",663757,657277,NaN,foul,...,3.44,0.99,-0.99,17.7,28.842199,-21.231754,35.413480,30.564350,38.876457,0
716788,CH,2026-03-25,87.1,-2.04,4.81,"Webb, Logan",663757,657277,NaN,foul,...,3.53,1.11,-1.11,18.1,38.922414,9.191064,33.248133,38.870183,29.350390,0


Count the number of NA rows for each column in our dataset where there is at least one NA row.

In [7]:
na_summary = (
    clean_df.isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
    .reset_index()
    .set_axis(['column_name', 'na_count'], axis=1)
)

print(na_summary.to_string())

                                 column_name  na_count
0                                      sv_id    338905
1                                     umpire    338905
2                        tfs_zulu_deprecated    338905
3                                   spin_dir    338905
4                       spin_rate_deprecated    338905
5                     break_angle_deprecated    338905
6                    break_length_deprecated    338905
7                             tfs_deprecated    338905
8                                      on_3b    304387
9                                      on_2b    271814
10                             miss_distance    262323
11                                     on_1b    232155
12            estimated_slg_using_speedangle    216823
13             estimated_ba_using_speedangle    216823
14                        launch_speed_angle    214805
15                                      hc_x    214356
16                                      hc_y    214356
17        

In [8]:
print(f'There are {len(clean_df.loc[0]) - len(na_summary)} columns with no NA rows (data exists in every row).')

There are 50 columns with no NA rows (data exists in every row).


The statcast CSV search documentation can be found at [this link](https://baseballsavant.mlb.com/csv-docs). As a team, we decided to drop all columns with all NA rows (`na_count = 338905` in the table above). 

In [9]:
clean_df = clean_df.loc[:, clean_df.isna().sum() < len(clean_df)]

The `on_1b`, `on_2b`, and `on_3b` columns contain integers that represent the IDs of the players who are on that base during the specific pitch, so NA rows represent cases where no player is on base. Since we do not care about the specific player on base but do care about the presence of a baserunner, we choose to replace the ID's with 1 and use 0 to represent the absence of a baserunner.

In [10]:
clean_df['on_1b'] = clean_df['on_1b'].notna().astype(int)
clean_df['on_2b'] = clean_df['on_2b'].notna().astype(int)
clean_df['on_3b'] = clean_df['on_3b'].notna().astype(int)
np.sum(clean_df[['on_1b', 'on_2b', 'on_3b']].isna())

np.int64(0)

We made a choice as a team to simply drop the following columns with NA values because they do not pertain to the goal of this analysis.

In [11]:
excluded_cols = ['miss_distance', 'estimated_slg_using_speedangle', 'estimated_ba_using_speedangle', 'launch_speed_angle',
                 'hc_x', 'hc_y', 'bb_type', 'hit_location', 'estimated_woba_using_speedangle', 'woba_denom', 'des',
                 'babip_value', 'woba_value', 'events', 'iso_value', 'launch_speed', 'hit_distance_sc', 'launch_angle', 
                 'hyper_speed', 'intercept_ball_minus_batter_pos_x_inches', 'intercept_ball_minus_batter_pos_y_inches', 
                 'batter_days_since_prev_game', 'batter_days_until_next_game', 'pitcher_days_since_prev_game',
                 'pitcher_days_until_next_game']

In [12]:
excluded_df = clean_df.drop(columns = excluded_cols)
excluded_df

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,description,zone,...,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,is_whiff
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,swinging_strike,13.0,...,1,1,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,1
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,swinging_strike,11.0,...,1,1,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,1
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,hit_into_play,3.0,...,1,3,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,0
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,hit_into_play,14.0,...,1,1,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,0
5,SL,2026-09-27,83.6,-2.27,5.87,"Imai, Tatsuya",695720,837227,foul,14.0,...,1,1,3.34,-0.05,-0.05,31.3,22.800268,-11.648310,37.518572,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
716781,SI,2026-03-25,92.0,-1.72,4.98,"Webb, Logan",592450,657277,foul,1.0,...,1,0,2.76,1.38,1.38,20.1,12.555138,-3.603324,36.148716,0
716786,FF,2026-03-25,93.6,-1.67,5.07,"Webb, Logan",663757,657277,swinging_strike,3.0,...,1,0,1.81,0.93,-0.93,22.0,15.601773,6.257514,29.116399,1
716787,CH,2026-03-25,86.9,-1.97,4.78,"Webb, Logan",663757,657277,foul,14.0,...,1,0,3.44,0.99,-0.99,17.7,28.842199,-21.231754,35.413480,0
716788,CH,2026-03-25,87.1,-2.04,4.81,"Webb, Logan",663757,657277,foul,8.0,...,1,0,3.53,1.11,-1.11,18.1,38.922414,9.191064,33.248133,0


We see that there are now $338905$ rows remaining, and we choose to drop all null rows as long as if that group composes less than 1% of our total dataset size. In other words, we want to make sure that we're only dropping rows if the columns that it corresponds to only has about 1% missing data, so dropping those rows allows for an easy treatment of the null values without the need for imputation or completely dropping the usable data.

In [13]:
p1_cutoff = int(np.round(0.01 * len(excluded_df)))

# Drop rows missing any column that has <= 1% overall missingness
na_counts = excluded_df.isna().sum()
cols_with_1pct_na = na_counts[(na_counts > 0) & (na_counts <= p1_cutoff)].index.tolist()
cleaned_df = excluded_df.dropna(subset=cols_with_1pct_na).copy()

# Drop rows whose exact null-column combination appears in < 1% of rows
patterns = cleaned_df.isna().apply(lambda r: tuple(cleaned_df.columns[r]), axis=1)
pattern_counts = patterns.value_counts()
cleaned_df = cleaned_df[patterns.isin(pattern_counts[pattern_counts >= p1_cutoff].index)].copy()
cleaned_df

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,description,zone,...,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,is_whiff
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,swinging_strike,13.0,...,1,1,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,1
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,swinging_strike,11.0,...,1,1,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,1
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,hit_into_play,3.0,...,1,3,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,0
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,hit_into_play,14.0,...,1,1,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,0
5,SL,2026-09-27,83.6,-2.27,5.87,"Imai, Tatsuya",695720,837227,foul,14.0,...,1,1,3.34,-0.05,-0.05,31.3,22.800268,-11.648310,37.518572,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
716781,SI,2026-03-25,92.0,-1.72,4.98,"Webb, Logan",592450,657277,foul,1.0,...,1,0,2.76,1.38,1.38,20.1,12.555138,-3.603324,36.148716,0
716786,FF,2026-03-25,93.6,-1.67,5.07,"Webb, Logan",663757,657277,swinging_strike,3.0,...,1,0,1.81,0.93,-0.93,22.0,15.601773,6.257514,29.116399,1
716787,CH,2026-03-25,86.9,-1.97,4.78,"Webb, Logan",663757,657277,foul,14.0,...,1,0,3.44,0.99,-0.99,17.7,28.842199,-21.231754,35.413480,0
716788,CH,2026-03-25,87.1,-2.04,4.81,"Webb, Logan",663757,657277,foul,8.0,...,1,0,3.53,1.11,-1.11,18.1,38.922414,9.191064,33.248133,0


We look at the remaining null rows and see what exact columns are missing data. This will tell us whether there are many columns with issues that are dispersed throughout the dataset, or if there's only a few culprit columns that make up our null rows.

In [14]:
null_patterns = (
    cleaned_df.isna()
    .apply(lambda row: tuple(cleaned_df.columns[row]), axis=1)
    .value_counts()
    .reset_index()
)
null_patterns.columns = ['null_columns', 'row_count']
null_patterns

,null_columns,row_count
0,(),329136
1,"(bat_speed, swing_length, attack_angle, attack...",7946


In [15]:
null_patterns.values

array([[(), 329136],
       [('bat_speed', 'swing_length', 'attack_angle', 'attack_direction', 'swing_path_tilt'),
        7946]], dtype=object)

About 8000 rows of data are simply missing the above 5 metrics. As a team, we decided to simply drop this data since it represents less than 3% of our entire dataset and saves us the problem of imputing data, which can introduce bias into the measures and our resulting analysis.

In [16]:
# Drop the 8000 rows with the missing values, so we don't have to deal with imputation at all
final_df = cleaned_df.dropna()
print('Number of NA rows left:', np.sum(final_df.isna()))
final_df.head()

Number of NA rows left: 0


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,description,zone,...,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,is_whiff
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,swinging_strike,13.0,...,1,1,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,1
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,swinging_strike,11.0,...,1,1,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,1
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,hit_into_play,3.0,...,1,3,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,0
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,hit_into_play,14.0,...,1,1,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,0
5,SL,2026-09-27,83.6,-2.27,5.87,"Imai, Tatsuya",695720,837227,foul,14.0,...,1,1,3.34,-0.05,-0.05,31.3,22.800268,-11.648310,37.518572,0


In [17]:
# Check data types of remaining columns
final_df.dtypes.value_counts()

int64      38
float64    35
str        14
Name: count, dtype: int64

In [18]:
# Output the names of the remaining columns
final_df.columns

Index(['pitch_type', 'game_date', 'release_speed', 'release_pos_x',
       'release_pos_z', 'player_name', 'batter', 'pitcher', 'description',
       'zone', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team',
       'type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x',
       'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning',
       'inning_topbot', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top',
       'sz_bot', 'effective_speed', 'release_spin_rate', 'release_extension',
       'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5',
       'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y',
       'at_bat_number', 'pitch_number', 'pitch_name', 'home_score',
       'away_score', 'bat_score', 'fld_score', 'post_away_score',
       'post_home_score', 'post_bat_score', 'post_fld_score',
       'if_fielding_alignment', 'of_fielding_alignment', 'spin_axis',
       'delta_home_win_exp', 'delta_run_exp', 'bat_speed', 'swing_lengt

In [19]:
# Export this dataset
final_df.to_csv('final_data.csv')

We now move towards setting up the dataset for the algorithms we will run (and analysis), which means we can remove even more columns that represent extraneous information that either does not pertain to our analysis or simply overlaps with other columns.

In [20]:
# Output the names of the non-numeric columns in our dataset
non_numeric_cols = final_df.select_dtypes(exclude='number').columns.tolist()
non_numeric_cols

['pitch_type',
 'game_date',
 'player_name',
 'description',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'inning_topbot',
 'pitch_name',
 'if_fielding_alignment',
 'of_fielding_alignment']

In [21]:
# Ascertain whether `pitch_type` and `pitch_name` values match up for every row
cleaned_df[['pitch_type', 'pitch_name']].value_counts().reset_index(name='count')

,pitch_type,pitch_name,count
0,FF,4-Seam Fastball,106624
1,SI,Sinker,53811
2,SL,Slider,44890
3,CH,Changeup,39746
4,FC,Cutter,27962
5,ST,Sweeper,26405
6,CU,Curveball,18151
7,FS,Split-Finger,11704
8,KC,Knuckle Curve,5224
9,SV,Slurve,1318


In [22]:
# Drop further unnecessary columns
analysis_df = final_df.drop(columns = ['fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7',
                                       'fielder_8', 'fielder_9', 'game_year', 'description', 'type', 'game_type', 
                                       'post_away_score', 'post_home_score', 'post_bat_score', 'post_fld_score', 
                                       'age_pit_legacy', 'age_bat_legacy']).copy()
analysis_df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,zone,stand,...,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,is_whiff
0,SL,2026-09-27,85.9,-2.24,5.85,"Imai, Tatsuya",672016,837227,13.0,R,...,1,1,2.60,0.22,0.22,30.6,29.219715,-44.217358,37.718503,1
1,FF,2026-09-27,95.2,-2.15,5.56,"Imai, Tatsuya",672016,837227,11.0,R,...,1,1,1.58,1.19,1.19,28.9,-1.752138,11.651549,29.176004,1
3,SL,2026-09-27,83.7,-2.23,5.83,"Imai, Tatsuya",811965,837227,3.0,R,...,1,3,2.92,0.22,0.22,31.3,8.361628,-6.452138,25.711833,0
4,SL,2026-09-27,84.3,-2.25,5.83,"Imai, Tatsuya",695720,837227,14.0,R,...,1,1,3.22,0.33,0.33,32.5,9.680133,-1.572133,28.116071,0
5,SL,2026-09-27,83.6,-2.27,5.87,"Imai, Tatsuya",695720,837227,14.0,R,...,1,1,3.34,-0.05,-0.05,31.3,22.800268,-11.648310,37.518572,0


In [23]:
# Output remaining columns in 'analysis' dataset
analysis_df.columns

Index(['pitch_type', 'game_date', 'release_speed', 'release_pos_x',
       'release_pos_z', 'player_name', 'batter', 'pitcher', 'zone', 'stand',
       'p_throws', 'home_team', 'away_team', 'balls', 'strikes', 'pfx_x',
       'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b',
       'outs_when_up', 'inning', 'inning_topbot', 'vx0', 'vy0', 'vz0', 'ax',
       'ay', 'az', 'sz_top', 'sz_bot', 'effective_speed', 'release_spin_rate',
       'release_extension', 'game_pk', 'release_pos_y', 'at_bat_number',
       'pitch_number', 'pitch_name', 'home_score', 'away_score', 'bat_score',
       'fld_score', 'if_fielding_alignment', 'of_fielding_alignment',
       'spin_axis', 'delta_home_win_exp', 'delta_run_exp', 'bat_speed',
       'swing_length', 'delta_pitcher_run_exp', 'home_score_diff',
       'bat_score_diff', 'home_win_exp', 'bat_win_exp', 'age_pit', 'age_bat',
       'n_thruorder_pitcher', 'n_priorpa_thisgame_player_at_bat',
       'api_break_z_with_gravity', 'api_break_x_arm', 'a

## README: Guide to dropping additional columns

The `pitch_type` and `pitch_name` columns track the same thing, so when modeling, only one of them is needed! I also suggest dropping: \
1) `player_name`: not necessary for actual ML algos, but useful to keep (and simply exclude from the algo) for reflection
2) `game_date`: if we do a time-series thing, then could be useful, but if not, prime candidate for dropping
3) `batter`: we don't care who the batter really is (small sample bias)
4) `pitcher`: same as batter
5) `home_team` and `away_team`: could be interesting to see how these correlate, but one-hot encoding introduces an additional $29*2 = 58$ columns from keeping these, so could consider dropping.
6) `game_pk`: this is essentially an ID column, so we definitely should not use it during modeling. However, this means I would suggest **against** dropping the column.
7) `home_score`, `away_score`, `bat_score`, `fld_score`: not really sure if we care about the specific scores of these four groups, so could drop if deemed unnecessary.
8) `delta_home_win_exp`, `delta_run_exp`, `delta_pitcher_run_exp`: I doubt that we really care about whether a singular whiff vs contact leads to +/- win expectancy, and I doubt the results will be noticeable anyway.
9) `home_score_diff` and `bat_score_diff`: These two are important, but I think we might only care about one of them (I would say we probably would want to keep `bat_score_diff` to see whether whiff vs no whiff occurs more when the batting team is trailing, tied, or leading)
10) `home_win_exp` and `bat_win_exp`: Maybe we just keep the batting team one to see whether players whiff more when their team is leading, tied, or losing.


## What's next?
**We need to decide:**
1) Which columns would we like to drop?
2) Is there anything else we would specifically want to see that we need to engineer as a feature?
3) Is the data after all this work compatible with our pre-existing work, and can we continue with our plans using this format of data?